In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name

BASE_PATH = "/Volumes/supply_chain_dev/shipments/raw_files"
BRONZE_TABLE = "supply_chain_dev.shipments.bronze_shipment_events"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/bronze"

# Define schema explicitly — never let Auto Loader guess on production data
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("booking_id", StringType(), True),
    StructField("container_id", StringType(), True),
    StructField("carrier", StringType(), True),
    StructField("origin_port", StringType(), True),
    StructField("destination_port", StringType(), True),
    StructField("status", StringType(), True),
    StructField("previous_status", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("vessel_name", StringType(), True),
    StructField("commodity_type", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("estimated_arrival", StringType(), True),
    StructField("exception_reason", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("schema_version", StringType(), True)
])

print("Config ready!")
print(f"Source : {BASE_PATH}/raw/shipment_events/")
print(f"Target : {BRONZE_TABLE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Config ready!
Source : /Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/
Target : supply_chain_dev.shipments.bronze_shipment_events
Checkpoint: /Volumes/supply_chain_dev/shipments/raw_files/checkpoints/bronze


In [0]:
from pyspark.sql.functions import current_timestamp, col

(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PATH)
    .schema(schema)
    .load(f"{BASE_PATH}/raw/shipment_events/")
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

print("Bronze ingestion complete!")

Bronze ingestion complete!


In [0]:
df = spark.table(BRONZE_TABLE)
print(f"Total records in Bronze table: {df.count()}")
display(df.select(
    "booking_id", "carrier", "origin_port",
    "destination_port", "status", "ingested_at", "source_file"
).limit(10))

Total records in Bronze table: 466


booking_id,carrier,origin_port,destination_port,status,ingested_at,source_file
EVG330035022,Evergreen,NLRTM,SGSIN,OUT_FOR_DELIVERY,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,BOOKING_CONFIRMED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,CARGO_RECEIVED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,VESSEL_DEPARTED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,IN_TRANSIT,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,PORT_ARRIVED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
CMA204078666,CMA CGM,HKHKG,CNSHA,CUSTOMS_HOLD,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
MSC959629660,Hapag-Lloyd,USLAX,NLRTM,BOOKING_CONFIRMED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
MSC959629660,Hapag-Lloyd,USLAX,NLRTM,CARGO_RECEIVED,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json
MSC959629660,Hapag-Lloyd,USLAX,NLRTM,null,2026-06-23T20:45:03.909Z,/Volumes/supply_chain_dev/shipments/raw_files/raw/shipment_events/batch1_file002.json


In [0]:
display(spark.sql("DESCRIBE DETAIL supply_chain_dev.shipments.bronze_shipment_events"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,debcdad7-bc87-48bb-8db8-5efb84e42477,supply_chain_dev.shipments.bronze_shipment_events,null,,2026-06-23T20:44:54.303Z,2026-06-23T20:45:19.000Z,List(),List(),1,24691,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-cfb7f58c-9e9a-4554-be7f-16dd2ab4ca23, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-61267a55-9993-4b0a-90da-b37cf56d3cae)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
display(spark.sql("DESCRIBE HISTORY supply_chain_dev.shipments.bronze_shipment_events"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-23T20:45:19.000Z,8479271986133632,ravigroverus@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> bde2f359-01fe-42b6-8ebd-cdb67fdf01cd, epochId -> 0, statsOnLoad -> true)",null,List(1175587767927121),e41d381a-5ad6-4ab1-bdc4-c7d5e25277d2,0623-204258-8vs7ytvj-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 466, numOutputBytes -> 24691, numAddedFiles -> 1)",null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
0,2026-06-23T20:44:56.000Z,8479271986133632,ravigroverus@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-cfb7f58c-9e9a-4554-be7f-16dd2ab4ca23"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-61267a55-9993-4b0a-90da-b37cf56d3cae""}, statsOnLoad -> false)",null,List(1175587767927121),e41d381a-5ad6-4ab1-bdc4-c7d5e25277d2,0623-204258-8vs7ytvj-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.2.x-aarch64-photon-scala2.13
